In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Sheet1"
series_col = "y"

df = pd.read_excel(file_path, sheet_name=sheet_name)
y = df[series_col].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "season_len": 12,      # int: 季节长度
    "forecast_steps": 6    # int: 预测步数
}

n = len(y)
t = np.arange(n).reshape(-1, 1)
trend_model = LinearRegression().fit(t, y)
trend = trend_model.predict(t)

ratio = y / np.maximum(trend, 1e-8)
seasonal_index = np.zeros(params["season_len"])
counts = np.zeros(params["season_len"])
for i, r in enumerate(ratio):
    idx = i % params["season_len"]
    seasonal_index[idx] += r
    counts[idx] += 1
seasonal_index /= np.maximum(counts, 1)
seasonal_index /= np.mean(seasonal_index)

t_future = np.arange(n, n + params["forecast_steps"]).reshape(-1, 1)
trend_future = trend_model.predict(t_future)
y_pred = np.array([
    trend_future[i] * seasonal_index[(n + i) % params["season_len"]]
    for i in range(params["forecast_steps"])
])
print(y_pred)


In [ ]:
"""
季节指数预测模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "季节指数预测模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "y"  # TODO: 请填写[目标列名]，说明：按时间升序排列的数值列。
SEASON_LENGTH = 12  # TODO: 请填写[季节周期]，说明：月度数据填 12，季度数据填 4。
FORECAST_STEPS = 12  # TODO: 请填写[预测期数]，说明：通常为一个或几个周期。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # 数据需按时间升序排列，周期长度例如月度季度性为 12，季度数据年度周期为 4。
    y = data[TARGET_COLUMN].to_numpy(dtype=float)
    t = np.arange(len(y))
    trend = np.polyval(np.polyfit(t, y, deg=1), t)
    ratio = y / trend

    # 按周期位置计算季节指数，并标准化到均值为 1。
    seasonal = np.array([ratio[i::SEASON_LENGTH].mean() for i in range(SEASON_LENGTH)])
    seasonal = seasonal / seasonal.mean()

    future_t = np.arange(len(y), len(y) + FORECAST_STEPS)
    future_trend = np.polyval(np.polyfit(t, y, deg=1), future_t)
    future_season = np.array([seasonal[i % SEASON_LENGTH] for i in future_t])
    forecast = future_trend * future_season
    result = pd.DataFrame({"预测期": future_t + 1, "预测值": forecast})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
